In [1]:
from collections import defaultdict


# =====================================
# Step 1 : Choose Text Corpus
# =====================================

corpus = [
    "low",
    "lower",
    "lowest",
    "new",
    "newer",
    "widest"
]


# =====================================
# Step 2 & 3 : Preprocess Text
# =====================================

corpus = [word.lower().strip() for word in corpus]


# =====================================
# Step 4 & 5 :
# Split Words into Characters + </w>
# Create Word Frequency Dictionary
# =====================================

vocab = defaultdict(int)

for word in corpus:
    characters = " ".join(list(word)) + " </w>"
    vocab[characters] += 1


print("Initial Vocabulary")
print(vocab)



# =====================================
# Step 6 & 7 :
# Find Adjacent Symbol Pair Frequencies
# =====================================

def get_pair_frequency(vocab):

    pairs = defaultdict(int)

    for word, freq in vocab.items():

        symbols = word.split()

        for i in range(len(symbols)-1):

            pair = (symbols[i], symbols[i+1])

            pairs[pair] += freq

    return pairs



# =====================================
# Step 9 & 10 :
# Merge Selected Pair
# =====================================

def merge_pair(pair, vocab):

    new_vocab = {}

    old_pair = " ".join(pair)
    new_token = "".join(pair)

    for word in vocab:

        new_word = word.replace(
            old_pair,
            new_token
        )

        new_vocab[new_word] = vocab[word]

    return new_vocab



# =====================================
# Step 11 & 12 :
# Repeat Merge and Store Rules
# =====================================

num_merges = 10

merge_rules = []


for i in range(num_merges):

    pair_frequency = get_pair_frequency(vocab)


    if len(pair_frequency) == 0:
        break


    best_pair = max(
        pair_frequency,
        key=pair_frequency.get
    )


    print("\nMerge", i+1)
    print("Selected Pair :", best_pair)
    print("Frequency :", pair_frequency[best_pair])


    merge_rules.append(best_pair)


    vocab = merge_pair(
        best_pair,
        vocab
    )



# =====================================
# Step 13 :
# Build Final Vocabulary
# =====================================

final_vocab = set()


for word in vocab:

    for token in word.split():

        final_vocab.add(token)



print("\n============================")
print("Final Vocabulary")
print("============================")


for token in sorted(final_vocab):

    print(token)



print("\nTotal Vocabulary Size :",len(final_vocab))



# =====================================
# Create Token IDs
# =====================================

token_to_id = {}

id_to_token = {}


for index, token in enumerate(sorted(final_vocab)):

    token_to_id[token] = index

    id_to_token[index] = token



# =====================================
# Step 14 :
# Encoder Function
# =====================================

def encode(word):

    tokens = list(word.lower()) + ["</w>"]


    for pair in merge_rules:

        i = 0

        while i < len(tokens)-1:

            if tokens[i] == pair[0] and tokens[i+1] == pair[1]:

                tokens[i] = tokens[i] + tokens[i+1]

                del tokens[i+1]

            else:

                i += 1


    token_ids = []

    for token in tokens:

        token_ids.append(
            token_to_id.get(token,-1)
        )


    return tokens, token_ids



# =====================================
# Step 15 :
# Decoder Function
# =====================================

def decode(tokens):

    word = ""

    for token in tokens:

        token = token.replace("</w>","")

        word += token


    return word



# =====================================
# Step 16 & 17 :
# Testing
# =====================================

test_words = [
    "low",
    "lower",
    "lowest",
    "newer",
    "widest"
]


print("\n============================")
print("Encoding and Decoding")
print("============================")


for word in test_words:

    tokens, ids = encode(word)

    decoded_word = decode(tokens)


    print("\nOriginal Word :",word)

    print("Tokens :",tokens)

    print("Token IDs :",ids)

    print("Decoded :",decoded_word)



print("\n============================")
print("Learned Merge Rules")
print("============================")


for i, rule in enumerate(merge_rules):

    print(i+1, ":", rule)

Initial Vocabulary
defaultdict(<class 'int'>, {'l o w </w>': 1, 'l o w e r </w>': 1, 'l o w e s t </w>': 1, 'n e w </w>': 1, 'n e w e r </w>': 1, 'w i d e s t </w>': 1})

Merge 1
Selected Pair : ('l', 'o')
Frequency : 3

Merge 2
Selected Pair : ('lo', 'w')
Frequency : 3

Merge 3
Selected Pair : ('low', 'e')
Frequency : 2

Merge 4
Selected Pair : ('r', '</w>')
Frequency : 2

Merge 5
Selected Pair : ('s', 't')
Frequency : 2

Merge 6
Selected Pair : ('st', '</w>')
Frequency : 2

Merge 7
Selected Pair : ('n', 'e')
Frequency : 2

Merge 8
Selected Pair : ('ne', 'w')
Frequency : 2

Merge 9
Selected Pair : ('low', '</w>')
Frequency : 1

Merge 10
Selected Pair : ('lowe', 'r</w>')
Frequency : 1

Final Vocabulary
</w>
d
e
i
low</w>
lowe
lower</w>
new
r</w>
st</w>
w

Total Vocabulary Size : 11

Encoding and Decoding

Original Word : low
Tokens : ['low</w>']
Token IDs : [4]
Decoded : low

Original Word : lower
Tokens : ['lower</w>']
Token IDs : [6]
Decoded : lower

Original Word : lowest
Tokens : [